# 07 · 사이트 이동

**질문:** 학습에서 사이트 하나를 통째로 제외하면, 그 사이트의 성능과 오경보는 어떻게 달라지는가?

02의 `data/results/features.csv`와 01의 `protocol.json`을 읽는다. 사이트마다 Cat28을 새로 학습하고, **나머지 사이트의 검증 집합에서만** seed·경보 임계값·확률 보정기를 정한다.

이 실험은 전체 특징 표에서 사이트별 분할을 새로 만든다. 01의 기존 train/validation/test 분할을 평가에 사용하지 않으므로, 03–06의 고정 test 성능과 별개의 분석이다. 결과의 `split` 열은 원래 분할 이력이고, `held_site`가 이번 평가 사이트다.

출력 기준은 `MIN_PUBLIC_MOTHERS = 5`다. 02의 특징 그룹·사이트 표와 07의 사이트별 표·그림은 서로 다른 산모가 이보다 적으면 숨긴다. 내부 CSV와 학습·평가 계산은 전체 자료를 유지한다. 이 기준은 표시를 제한하는 기본값이며 기관의 반출 기준을 대신하지 않는다.

## 1. 입력과 설정

원본과 같이 모델 seed는 42·43·44, 검증 비율은 산모의 20%, 산모 bootstrap은 2,000회다. 산모 분할 seed는 01의 프로토콜을 따른다. 학습 반복 상한·깊이·학습률·조기 종료 조건도 유지한다.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from sklearn.linear_model import LogisticRegression

RESULTS = Path("data/results")
SEEDS = (42, 43, 44)
VALIDATION_FRACTION = 0.2
TREE_ITERATIONS = 1000
TREE_PATIENCE = 80
THREADS = 2
N_BOOTSTRAP = 2000
BOOTSTRAP_SEED = 42
CALIBRATION_SEED = 42
MIN_PUBLIC_MOTHERS = 5  # Display rule only; evaluation still includes every eligible site.
ID_TYPES = {key: str for key in ("row_id", "record_id", "mother_id", "site")}


### 출력에 남기는 오류 정보

오류는 고정된 분류와 건수로만 요약한다. `other_error`의 상세 원인은 내부 오류 파일에서 확인한다. 건수는 오류 내역 수이며, 같은 기록이 여러 번 포함될 수 있다.

In [ ]:
def issue_category(reason):
    # Match known phrases, but return ONLY fixed labels: never echo an exception.
    # Example: "FileNotFoundError: /private/DEMO.json" -> "missing_input".
    text = str(reason)
    groups = {
        "missing_input": ("FileNotFoundError", "No such file", "[Errno 2]"),
        "missing_dependency": ("No module named", "cannot import name", "ImportError"),
        "fhr_missing": ("FHR 결측률",),
        "invalid_wave": ("FHR과 TOCO", "측정 간격", "FHR은 0 이상", "신호 길이"),
        "invalid_bbox": ("Bbox",),
        "invalid_id": ("ID가", "산모 ID", "산모 목록", "행 정보 불일치", "구간 정보 불일치"),
        "invalid_labels": ("라벨이 올바르지", "라벨 수", "라벨·점수"),
        "no_usable_data": ("구간이 없습니다", "비었", "남은 산모가 없습니다"),
        "split_unavailable": ("분할", "양성·음성", "정상·비정상", "least populated class", "test_size"),
        "checkpoint_mismatch": ("체크포인트", "CNN 입력·설정"),
        "io_error": ("PermissionError", "Permission denied", "No space left", "OSError"),
        "invalid_value": ("ValueError", "TypeError", "유한", "could not convert", "ufunc"),
    }
    for category, phrases in groups.items():
        if any(phrase in text for phrase in phrases):
            return category
    return "other_error"

In [ ]:
def show_issue_counts(reasons):
    # Count issue entries, not unique people; one record may have several entries.
    categories = pd.Series([issue_category(reason) for reason in reasons], dtype="string")
    counts = categories.value_counts().rename_axis("category").to_frame("count")
    print("제외·오류 내역 수 =", len(categories))
    # Full messages, IDs, paths, and jobs stay in the internal CSV files.
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(counts)

In [ ]:
# Feature order is the same Cat28 order used in 02 and 04.
CAT28 = [
    "fhr_mean", "fhr_min", "fhr_max", "fhr_sd", "stv", "brady_frac", "tachy_frac",
    "n_decel", "decel_max_depth", "decel_time_frac", "n_accel",
    "toco_mean", "toco_max", "toco_sd", "n_contractions",
    "ft_corr0", "ft_corr_min", "ft_corr_min_lag_s", "figo_baseline", "figo_baseline_var",
    "n_early_decel", "n_late_decel", "n_variable_decel", "n_severe_decel", "n_prolonged_decel",
    "hist_width", "hist_median", "hist_mode",
]
PREDICTION_COLUMNS = [
    "row_id", "record_id", "mother_id", "site", "target", "split",
    "record_all_normal", "signal_index", "score", "threshold",
]
METRICS = (
    "AP", "AUROC", "Brier", "sensitivity", "specificity", "PPV",
    "normal_record_alarm", "alarms_per_normal_hour",
)
METRIC_COLUMNS = ["metric", "point", "low", "high", "valid_bootstrap", "mothers", "segments"]
COHORT_COLUMNS = ["site", "windows", "mothers", "positive_mothers", "prevalence"]

In [ ]:
frame = pd.DataFrame()
protocol = {}
input_error = None
issues = []
try:
    protocol = json.loads((RESULTS / "protocol.json").read_text(encoding="utf-8"))
    # IDs are strings; "00DEMO527183" is a fictional example with leading zeros.
    frame = pd.read_csv(RESULTS / "features.csv", dtype=ID_TYPES)
    required = PREDICTION_COLUMNS[:-2] + CAT28
    missing_columns = sorted(set(required) - set(frame.columns))
    if missing_columns:
        raise ValueError("특징 표에 필요한 열이 없습니다: " + str(missing_columns))
    if frame.empty or frame["row_id"].duplicated().any():
        raise ValueError("특징 표가 비었거나 구간 ID가 중복됩니다")
    if frame[list(ID_TYPES)].isna().any().any() or not frame["target"].isin([0, 1]).all():
        raise ValueError("ID·사이트·라벨에 결측 또는 잘못된 값이 있습니다")
    split_seed = int(protocol["split_seed"])
except (OSError, ValueError, KeyError, TypeError) as error:
    # Missing input is recorded; later cells save empty outputs for this run.
    input_error = f"{type(error).__name__}: {error}"
    issues.append(dict(site="", stage="input", reason=input_error))
    print('입력을 이용할 수 없습니다:', '[details omitted from notebook output]')

In [ ]:
if input_error is None:
    print("feature table shape =", frame.shape)
    print("sites =", sorted(frame["site"].unique()))
    print("source split seed =", split_seed)
    # Example: a row originally marked test can enter source training for another site.

## 2. 평가 사이트와 학습 가능한 산모를 나눈다

평가 집합은 해당 사이트의 구간이다. 그 사이트에 등장한 산모는 **다른 사이트에 있는 구간까지 모두** source에서 제외한다. source는 이번에 학습·검증에 사용할 후보 집합을 뜻한다.

In [ ]:
def hold_out_site(data, site):
    held = data.loc[data["site"].eq(site)]
    held_mothers = held["mother_id"].unique()
    # Example: M1 has rows at A and B; holding out A excludes both from source.
    # Only M1's A rows enter held; M1's B rows enter neither set.
    source = data.loc[~data["mother_id"].isin(held_mothers)]
    return source, held

In [ ]:
example_rows = pd.DataFrame({
    "row_id": ["R1:0", "R2:0", "R3:0"],
    "mother_id": ["M1", "M1", "M2"],
    "site": ["A", "B", "B"],
})
example_source, example_held = hold_out_site(example_rows, "A")
# held = [R1:0], source = [R3:0]; R2:0 is excluded because its mother is M1.
print("held rows =", example_held["row_id"].tolist())
print("source rows =", example_source["row_id"].tolist())

In [ ]:
def split_source_mothers(source, seed):
    # Example: one mother's labels [0, 1, 0] -> mother label 1.
    # Stratify on this label, keeping repeat records and twins together.
    mother_labels = source.groupby("mother_id")["target"].max()
    train_ids, validation_ids = train_test_split(
        mother_labels.index.to_numpy(),
        test_size=VALIDATION_FRACTION,
        stratify=mother_labels.to_numpy(),
        random_state=seed,
    )
    # Example: 100 source mothers -> 80 train mothers and 20 validation mothers.
    # Window counts need not be 80:20; mothers have different numbers of windows.
    train = source.loc[source["mother_id"].isin(train_ids)]
    validation = source.loc[source["mother_id"].isin(validation_ids)]
    return train, validation

In [ ]:
if input_error is None:
    sample_site = sorted(frame["site"].unique())[0]
    sample_source, sample_held = hold_out_site(frame, sample_site)
    print("held site =", sample_site)
    print("held windows / mothers =", len(sample_held), sample_held["mother_id"].nunique())
    print("source windows / mothers =", len(sample_source), sample_source["mother_id"].nunique())
    # This example reports the split without training a model.
    try:
        sample_train, sample_validation = split_source_mothers(sample_source, split_seed)
        print("train mothers =", sample_train["mother_id"].nunique())
        print("validation mothers =", sample_validation["mother_id"].nunique())
        overlap = set(sample_held["mother_id"]) & set(sample_source["mother_id"])
        print("held/source mother overlap =", len(overlap))
    except ValueError as error:
        print('이 사이트는 분할할 수 없습니다:', '[details omitted from notebook output]')

## 3. source 검증 AP로 모델을 고른다

같은 산모 분할에서 세 모델 seed를 모두 학습한다. 검증 AP가 가장 큰 모델을 선택하고, 동점이면 작은 seed를 고른다. 하나라도 실패하면 해당 사이트를 미완료로 남긴다. 평가 사이트의 라벨은 이 선택에 사용하지 않는다.

In [ ]:
def fit_catboost(train, validation, seed):
    from catboost import CatBoostClassifier

    model = CatBoostClassifier(
        iterations=TREE_ITERATIONS, depth=6, learning_rate=0.05,
        loss_function="Logloss", eval_metric="PRAUC", auto_class_weights="SqrtBalanced",
        random_seed=seed, thread_count=THREADS, allow_writing_files=False, verbose=False,
    )
    # Training fits trees; source validation decides when to stop adding trees.
    # Example: 80 rounds without an improved validation metric -> early stop.
    model.fit(
        train[CAT28], train["target"],
        eval_set=(validation[CAT28], validation["target"]),
        early_stopping_rounds=TREE_PATIENCE,
    )
    return model

In [ ]:
def select_source_model(train, validation):
    candidates = []
    for seed in SEEDS:
        model = fit_catboost(train, validation, seed)
        validation_scores = model.predict_proba(validation[CAT28])[:, 1]
        validation_ap = average_precision_score(validation["target"], validation_scores)
        candidates.append(dict(
            seed=seed, model=model, scores=validation_scores, AP=validation_ap,
        ))
    # Example: AP [0.70, 0.72, 0.72] for seeds [42, 43, 44] -> select seed 43.
    # Selection happens only after all three candidates finish successfully.
    selected = max(candidates, key=lambda candidate: (candidate["AP"], -candidate["seed"]))
    return selected

## 4. source에서 경보 기준과 확률 보정을 정한다

검증 음성의 90% 이상을 경보하지 않는 임계값을 구한다. 같은 검증 집합에서 Platt 보정기를 적합하고, 보정 점수의 임계값도 다시 정한다. 평가 사이트에서는 이 두 기준을 그대로 적용한다. **검증 특이도 90%가 평가 사이트에서도 보장되는 것은 아니다.**

In [ ]:
def threshold90(labels, scores):
    negative_scores = np.sort(np.asarray(scores)[np.asarray(labels) == 0])
    if not len(negative_scores):
        raise ValueError("검증 음성이 없어 임계값을 정할 수 없습니다")
    # Example: 10 negatives -> ceil(0.9 * 10) = 9th value -> index 8.
    boundary_index = int(np.ceil(0.9 * len(negative_scores))) - 1
    # A high-scoring positive never enters this negative-score order statistic.
    boundary = negative_scores[boundary_index]
    # Move above the boundary because score == threshold triggers an alarm.
    # Example: boundary 0.8 -> threshold 0.8000000000000002.
    return np.nextafter(boundary, np.inf)

In [ ]:
# Ten negative scores: the 9th is 0.8, so only 0.9 should trigger an alarm.
example_scores = np.arange(10) / 10
example_threshold = threshold90(np.zeros(10), example_scores)
print("threshold =", example_threshold)
print("negative alarms =", (example_scores >= example_threshold).tolist())
print("specificity =", np.mean(example_scores < example_threshold))

In [ ]:
def logit_scores(values):
    # Example: p=0.5 -> logit=0; p=0 or 1 is clipped before taking logarithms.
    # Example: p=0 becomes 0.000001, yielding a finite logit near -13.82.
    clipped = np.clip(np.asarray(values), 1e-6, 1 - 1e-6)
    logits = np.log(clipped / (1 - clipped))
    # LogisticRegression expects (windows, features): one logit feature here.
    return logits.reshape(-1, 1)

In [ ]:
def make_prediction(held, scores, threshold):
    # One row describes one held-site window, in the original feature-table order.
    prediction = held[PREDICTION_COLUMNS[:-2]].copy()
    prediction["score"] = np.asarray(scores, dtype=float)
    prediction["threshold"] = threshold
    if prediction["row_id"].duplicated().any() or not np.isfinite(prediction["score"]).all():
        raise ValueError("예측 구간 ID가 중복되거나 점수가 유한하지 않습니다")
    # Example: score=0.7, threshold=0.6 -> alarm; scores equal to threshold also alarm.
    # Keep the original CSV contract: derive alarms when evaluating, not as a column.
    return prediction

In [ ]:
def site_predictions(validation, held, selected):
    validation_scores = selected["scores"]
    held_scores = selected["model"].predict_proba(held[CAT28])[:, 1]
    raw_threshold = threshold90(validation["target"], validation_scores)
    raw = make_prediction(held, held_scores, raw_threshold)

    # Fit one logistic curve using source validation labels and logits only.
    # Held-site labels never enter fit() or threshold90().
    calibrator = LogisticRegression(random_state=CALIBRATION_SEED)
    calibrator.fit(logit_scores(validation_scores), validation["target"])
    calibrated_validation = calibrator.predict_proba(logit_scores(validation_scores))[:, 1]
    calibrated_held = calibrator.predict_proba(logit_scores(held_scores))[:, 1]
    calibrated_threshold = threshold90(validation["target"], calibrated_validation)
    calibrated = make_prediction(held, calibrated_held, calibrated_threshold)
    # A positive slope preserves rank; it does not guarantee improved target calibration.
    slope = float(calibrator.coef_[0, 0])
    return raw, calibrated, slope

## 5. 사이트 안에서 성능과 경보를 센다

AP·AUROC·Brier와 다섯 운영 지표를 계산한다. 정상 기록 경보율의 분모는 **원래 모든 구간이 정상인 기록 수**다. 정상 시간당 경보의 분모는 그런 기록의 유지된 5분 구간 수 ÷ 12이며, 연속 경보를 하나의 사건으로 합치지 않는다.

아래 정의는 06과 같다. 산모를 반복 추출한 횟수로 구간과 기록에 가중치를 주므로, 같은 산모의 반복 기록·쌍태아가 함께 움직인다.

In [ ]:
def prepare_metrics(prediction):
    labels = prediction["target"].to_numpy(dtype=int)
    scores = prediction["score"].to_numpy(dtype=float)
    if not len(prediction) or set(labels) - {0, 1} or not np.isfinite(scores).all():
        raise ValueError("평가 라벨·점수가 올바르지 않거나 평가 집합이 비었습니다")

    # Example: mother IDs [A, A, B] -> mother_index [0, 0, 1].
    mothers, mother_index = np.unique(prediction["mother_id"], return_inverse=True)
    # Example: scores [0.2, 0.9, 0.5] -> descending row order [1, 2, 0].
    order = np.argsort(-scores, kind="stable")
    # Example: sorted scores [0.9, 0.9, 0.3] -> group ends [1, 2].
    # Tied scores are one threshold, independent of their row order.
    ends = np.r_[np.flatnonzero(np.diff(scores[order])), len(prediction) - 1]
    return dict(
        labels=labels, scores=scores, mothers=mothers,
        mother_index=mother_index, order=order, ends=ends,
    )

In [ ]:
def prepare_operating_data(prediction):
    data = prepare_metrics(prediction)
    thresholds = prediction["threshold"].to_numpy(dtype=float)
    if not np.isfinite(thresholds).all():
        raise ValueError("운영 평가에 유한한 임계값이 필요합니다")
    # Recompute from scores and thresholds, independent of the saved alarm column.
    data["alarm"] = data["scores"] >= thresholds
    # A record with an excluded abnormal window still has normal=False here.
    data["normal"] = prediction["record_all_normal"].to_numpy(dtype=bool)

    # Example: records [R1, R1, R2] -> record_index [0, 0, 1], first_rows [0, 2].
    _, first_rows, record_index = np.unique(
        prediction["record_id"], return_index=True, return_inverse=True,
    )
    # Example: two records from mother M1 both use M1's bootstrap count.
    data["record_mother"] = data["mother_index"][first_rows]
    data["normal_record"] = data["normal"][first_rows]
    # One alarming window is enough to mark the whole record as alarming.
    data["record_alarm"] = np.bincount(record_index, weights=data["alarm"]) > 0
    return data

In [ ]:
def rank_metrics(data, counts=None):
    if counts is None:
        # Point estimate: every mother appears once, so every window has weight 1.
        counts = np.ones(len(data["mothers"]))
    # Example: mother counts [2, 0] -> weights [2, 2, 0] for [A, A, B].
    weights = counts[data["mother_index"]]
    labels = data["labels"]
    # Denominators count weighted abnormal and normal *windows*.
    positive = np.sum(weights * labels)
    negative = np.sum(weights * (1 - labels))
    # Skip a resample containing only normal or only abnormal windows.
    if not positive or not negative:
        return None

    order, ends = data["order"], data["ends"]
    # Count windows at or above each distinct score threshold.
    # Example: distinct descending scores with labels [1, 0, 1] -> TP [0, 1, 1, 2].
    true_positive = np.r_[0, np.cumsum((weights * labels)[order])[ends]]
    false_positive = np.r_[0, np.cumsum((weights * (1 - labels))[order])[ends]]
    recall = true_positive / positive
    false_positive_rate = false_positive / negative
    predicted_positive = np.maximum(true_positive[1:] + false_positive[1:], 1e-30)
    # Example: descending labels [1, 0, 1, 0] -> AP = (1 + 2/3)/2 = 5/6.
    ap = np.sum(np.diff(recall) * true_positive[1:] / predicted_positive)
    # Those labels with distinct scores give AUROC=3/4.
    auc = np.sum(np.diff(false_positive_rate) * (recall[1:] + recall[:-1]) / 2)
    # Example: label 1, score 0.9 contributes (0.9 - 1)^2 = 0.01.
    brier = np.sum(weights * (data["scores"] - labels) ** 2) / weights.sum()
    return dict(AP=ap, AUROC=auc, Brier=brier)

In [ ]:
def operating_rates(data, counts):
    weights = counts[data["mother_index"]]
    labels, alarms = data["labels"], data["alarm"]
    positive = np.sum(weights * labels)
    negative = np.sum(weights * (1 - labels))
    true_alarm = np.sum(weights * labels * alarms)
    false_alarm = np.sum(weights * (1 - labels) * alarms)
    # Example: TP=1, FN=1 -> sensitivity=1/2; TP=1, FP=2 -> PPV=1/3.
    # Specificity = 1 - FP/negative windows; no alarms means undefined PPV.
    rates = dict(
        sensitivity=true_alarm / positive,
        specificity=1 - false_alarm / negative,
        PPV=true_alarm / (true_alarm + false_alarm) if true_alarm + false_alarm else np.nan,
    )
    # Each record inherits its mother's draw count; keep only normal records.
    record_weights = counts[data["record_mother"]] * data["normal_record"]
    # Count retained windows from all-normal records, not every target=0 window.
    normal_hours = np.sum(weights * data["normal"]) / 12
    # Example: 1 alarming record / 2 normal records -> 0.5.
    rates["normal_record_alarm"] = (
        np.sum(record_weights * data["record_alarm"]) / record_weights.sum()
        if record_weights.sum() else np.nan
    )
    # Example: 1 alarm in 4 normal windows -> 1 / (4/12) = 3 alarms per hour.
    rates["alarms_per_normal_hour"] = (
        np.sum(weights * data["normal"] * alarms) / normal_hours if normal_hours else np.nan
    )
    return rates

In [ ]:
def weighted_metrics(data, counts=None):
    if counts is None:
        # Point estimate: every mother is counted once.
        counts = np.ones(len(data["mothers"]))
    result = rank_metrics(data, counts)
    if result is None:
        # The original rule skips one-class draws for all eight metrics.
        return None
    result.update(operating_rates(data, counts))
    return result

In [ ]:
example_prediction = pd.DataFrame({
    "mother_id": ["A", "A", "B", "B"],
    "record_id": ["R1", "R1", "R2", "R2"],
    "target": [0, 0, 1, 0],
    "record_all_normal": [1, 1, 0, 0],
    "score": [0.8, 0.2, 0.9, 0.1],
    "threshold": [0.7, 0.7, 0.7, 0.7],
})
example_metrics = weighted_metrics(prepare_operating_data(example_prediction))
# R1 is the only all-normal record; one of its two windows alarms.
# normal_record_alarm = 1/1; alarms_per_normal_hour = 1/(2/12) = 6.
print("normal record alarm fraction =", example_metrics["normal_record_alarm"])
print("alarms per normal hour =", example_metrics["alarms_per_normal_hour"])
print("specificity =", example_metrics["specificity"])  # 2 of 3 negative windows are quiet.

## 6. 산모 bootstrap으로 신뢰구간을 만든다

각 사이트의 산모를 같은 수만큼 복원추출하고 2,000회 반복한다. 모델·보정기·임계값은 다시 학습하지 않는다. 양성 또는 음성이 없는 반복은 제외하며, PPV처럼 분모가 0인 지표는 그 지표에서만 제외한다. `valid_bootstrap`은 실제로 사용한 반복 수다.

In [ ]:
def resample_site_metrics(data, n_boot, seed):
    samples = {metric: [] for metric in METRICS}
    mother_count = len(data["mothers"])
    rng = np.random.default_rng(seed)
    for _ in range(n_boot):
        # Example: drawing mothers [A, A, C] -> counts [2, 0, 1] for [A, B, C].
        # Every record/window from A now has weight 2; B has weight 0.
        draws = rng.integers(mother_count, size=mother_count)
        counts = np.bincount(draws, minlength=mother_count)
        result = weighted_metrics(data, counts)
        if result is None:
            continue
        for metric in METRICS:
            # Example: no alarms -> PPV is undefined, but specificity remains usable.
            if np.isfinite(result[metric]):
                samples[metric].append(result[metric])
    return samples

In [ ]:
def site_intervals(prediction, n_boot=N_BOOTSTRAP, seed=BOOTSTRAP_SEED):
    data = prepare_operating_data(prediction)
    point = weighted_metrics(data)
    if point is None:
        raise ValueError("평가에는 정상·비정상 구간이 모두 필요합니다")
    samples = resample_site_metrics(data, n_boot, seed)
    rows = []
    for metric, value in point.items():
        low, high = np.nan, np.nan
        if samples[metric]:
            # Keep the middle 95%: quantiles 0.025 and 0.975 of valid draws.
            low, high = np.quantile(samples[metric], [0.025, 0.975])
        rows.append(dict(
            metric=metric, point=value, low=low, high=high,
            valid_bootstrap=len(samples[metric]),
            mothers=len(data["mothers"]), segments=len(prediction),
        ))
    return pd.DataFrame(rows)

## 7. 사이트마다 학습·선택·평가한다

한 사이트의 raw·Platt 평가를 모두 마친 뒤 최종 목록에 추가한다. 실패한 사이트는 이유를 남기고 다음 사이트로 넘어간다. 원본의 평가 방식은 유지하되, 실패한 사이트의 일부 결과나 이전 실행 파일이 완료 결과처럼 남지 않게 한다.

In [ ]:
def evaluate_site(source, held, split_seed):
    train, validation = split_source_mothers(source, split_seed)
    for name, subset in (("train", train), ("validation", validation), ("held", held)):
        if set(subset["target"].unique()) != {0, 1}:
            raise ValueError(name + "에 정상·비정상 구간이 모두 필요합니다")
    # The target site must share no mother with either source partition.
    if set(held["mother_id"]) & set(source["mother_id"]):
        raise ValueError("평가 사이트의 산모가 source에 포함되었습니다")

    selected = select_source_model(train, validation)
    raw, calibrated, slope = site_predictions(validation, held, selected)
    site = held["site"].iloc[0]
    seed = selected["seed"]
    site_metrics, site_rows = [], []
    for arm, prediction in (("raw", raw), ("platt", calibrated)):
        # Example: 8 metrics x 2 arms -> 16 metric rows for one completed site.
        metrics = site_intervals(prediction).assign(site=site, arm=arm, seed=seed)
        site_metrics.append(metrics)
        site_rows.append(prediction.assign(held_site=site, arm=arm, seed=seed))
    status = dict(
        site=site, status="complete", selected_seed=seed,
        source_mothers=source["mother_id"].nunique(), platt_slope=slope,
    )
    return site_metrics, site_rows, status

In [ ]:
metric_parts, prediction_parts = [], []
cohort_rows, status_rows = [], []
sites = sorted(frame["site"].unique()) if input_error is None else []
for site in sites:
    source, held = hold_out_site(frame, site)
    # Prevalence uses windows; positive_mothers counts mothers with any positive window.
    # Example: labels [0, 1, 1, 0] -> prevalence = 2/4 = 0.5.
    cohort_rows.append(dict(
        site=site, windows=len(held), mothers=held["mother_id"].nunique(),
        positive_mothers=held.loc[held["target"].eq(1), "mother_id"].nunique(),
        prevalence=held["target"].mean(),
    ))
    try:
        site_metrics, site_rows, site_status = evaluate_site(source, held, split_seed)
        # Commit both arms together; a failed site contributes neither arm.
        metric_parts.extend(site_metrics)
        prediction_parts.extend(site_rows)
        status_rows.append(site_status)
        print("LOSO", site, "complete", flush=True)
    except Exception as error:
        reason = str(error)
        status_rows.append(dict(site=site, status="unavailable", reason=reason))
        issues.append(dict(site=site, stage="site", reason=f"{type(error).__name__}: {error}"))
        print('LOSO', site, 'unavailable:', '[details omitted from notebook output]', flush=True)

In [ ]:
# Empty schemas replace stale results when this run has no completed site.
metrics = pd.DataFrame(columns=METRIC_COLUMNS + ["site", "arm", "seed"])
predictions = pd.DataFrame(columns=PREDICTION_COLUMNS + ["held_site", "arm", "seed"])
if metric_parts:
    metrics = pd.concat(metric_parts, ignore_index=True)
    predictions = pd.concat(prediction_parts, ignore_index=True)
inventory = pd.DataFrame(cohort_rows, columns=COHORT_COLUMNS)
statuses = pd.DataFrame(status_rows)
if statuses.empty:
    statuses = pd.DataFrame(columns=["site", "status", "selected_seed", "source_mothers", "platt_slope", "reason"])
issue_table = pd.DataFrame(issues, columns=["site", "stage", "reason"])
complete_count = sum(row["status"] == "complete" for row in status_rows)
print("completed sites =", complete_count, "/", len(sites))

if not issue_table.empty:
    print("issue count =", len(issue_table))


## 8. 사이트별 표와 그림을 읽는다

코호트 표는 사이트당 한 행, 지표 표는 사이트 × 보정 조건 × 지표당 한 행이다. 예측 표는 같은 구간을 raw·Platt 각각 한 번씩 담는다. 모든 결과에는 사이트별로 source에서 선택한 seed가 붙는다.

In [ ]:
# A site with 100 windows from 2 mothers is still below the display threshold.
public_inventory = inventory.loc[inventory["mothers"].ge(MIN_PUBLIC_MOTHERS)].copy()
public_site_codes = set(public_inventory["site"])
public_statuses = statuses.loc[statuses["site"].isin(public_site_codes)].drop(columns="reason", errors="ignore")
public_metrics = metrics.loc[metrics["site"].isin(public_site_codes)].copy()
print("공개 최소 산모 수 =", MIN_PUBLIC_MOTHERS)
print("숨긴 사이트 수 =", len(inventory) - len(public_inventory))
# Both raw and Platt arms, all eight metrics, and all interval columns are shown.
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(public_statuses)
    display(public_inventory)
    display(public_metrics)


In [ ]:
if not public_metrics.empty:
    raw_ap = public_metrics.loc[public_metrics["arm"].eq("raw") & public_metrics["metric"].eq("AP")].set_index("site")
    ap_table = public_inventory.set_index("site").join(raw_ap[["point", "low", "high"]])
    # AP depends on prevalence: read both bars for each site, not AP alone.
    ap_table[["prevalence", "point"]].rename(columns={"point": "AP"}).plot.bar(
        figsize=(10, 4), title="Unseen site: AP and prevalence",
    )
    plt.tight_layout()
    plt.show()


In [ ]:
if not public_metrics.empty:
    specificity = public_metrics.loc[public_metrics["arm"].eq("raw") & public_metrics["metric"].eq("specificity")]
    figure, axis = plt.subplots(figsize=(10, 4))
    for position, row in enumerate(specificity.itertuples()):
        # Vertical line: 95% mother-bootstrap interval; dot: observed site specificity.
        axis.vlines(position, row.low, row.high, color="C0")
        axis.scatter(position, row.point, color="C0")
    axis.set_xticks(range(len(specificity)), specificity["site"])
    # 0.90 was a SOURCE validation target, not a guaranteed held-site result.
    axis.axhline(0.9, color="gray", linestyle="--")
    axis.set(ylabel="Specificity", title="Source threshold applied to unseen sites")
    figure.tight_layout()
    plt.show()


## 9. 이번 실행의 결과를 저장한다

모두 `data/results/`에 저장한다. 원본의 네 CSV와 완료 사이트 수를 유지하고, 입력·사이트 오류 표를 추가한다. 학습한 모델은 저장하지 않는다.

| 파일 | 내용 |
|---|---|
| `07_site_cohorts.csv` | 평가 사이트별 구간 수·산모 수·기저율 |
| `07_site_metrics.csv` | raw·Platt 지표와 산모 bootstrap 신뢰구간 |
| `07_predictions.csv` | 평가 구간별 점수·임계값·사이트·seed |
| `07_status.csv` | 사이트별 완료 여부·선택 seed 또는 실패 이유 |
| `07_status.json` | 완료 수·전체 사이트 수·입력 상태 |
| `07_issues.csv` | 입력 또는 사이트 처리 오류 |

결과가 없으면 빈 열 구조로 갱신한다. `07_status.json`에서 이번 실행의 완료 여부를 먼저 확인한다.

In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)
metrics.to_csv(RESULTS / "07_site_metrics.csv", index=False)
inventory.to_csv(RESULTS / "07_site_cohorts.csv", index=False)
predictions.to_csv(RESULTS / "07_predictions.csv", index=False)
statuses.to_csv(RESULTS / "07_status.csv", index=False)
issue_table.to_csv(RESULTS / "07_issues.csv", index=False)

run_state = "unavailable"
if complete_count:
    run_state = "complete" if complete_count == len(sites) else "partial"
status = dict(
    complete=complete_count, total=len(sites), status=run_state, input_error=input_error,
)
(RESULTS / "07_status.json").write_text(
    json.dumps(status, ensure_ascii=False, indent=2), encoding="utf-8",
)
print('saved to =', RESULTS)
print('status =', status['status'])

## 해석의 범위

동일 자료 체계 안에서 사이트를 제외한 실험이다. 사이트 코드와 실제 병원의 일대일 대응은 확인되지 않았으므로 **독립 외부병원·전향적 검증으로 부르지 않는다.**

AP는 기저율과 함께 읽는다. 신뢰구간은 해당 사이트 산모를 다시 뽑았을 때의 변동이며, 새 병원 선택이나 모델 재학습의 변동을 포함하지 않는다. 표본이 작은 사이트는 `mothers`와 `valid_bootstrap`도 확인한다. 사이트 라벨로 임계값을 맞추지 않았으므로, source에서 정한 경보 기준이 이동했을 때의 변화를 볼 수 있다.

### 반출용 제외·오류 집계

개별 ID·경로·오류 전문을 제외한 분류별 건수다.

In [ ]:
show_issue_counts(issue_table["reason"])